# PH-05 Classification - AirGlobal Dataset

Notebook hoàn chỉnh cho phần PH-05 theo bảng phân công: baseline models với Stratified 5-Fold CV, XGBoost và LightGBM với Optuna tuning, Stacking Ensemble, SHAP, LIME, PDP/ICE, MLflow logging, calibration xác suất và lưu artifact phục vụ backend.

Phạm vi xử lý của notebook này là đọc output đã được tạo từ PH-03, huấn luyện các mô hình phân loại AQI Bucket, đánh giá trên validation/test set và lưu các kết quả phục vụ báo cáo. Notebook không đọc raw dataset, không tạo datalake và không dùng trực tiếp AQI/AQI Bucket/target làm feature để tránh data leakage.


## 1. Import thư viện

Cell này nạp các thư viện cần thiết cho quá trình huấn luyện, đánh giá, giải thích mô hình và lưu artifact. Một số thư viện như XGBoost, LightGBM, Optuna, SHAP, LIME và MLflow được xử lý theo hướng optional để notebook vẫn chạy được nếu môi trường chưa cài đầy đủ.


In [1]:
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


## 2. Cấu hình đường dẫn và chế độ chạy

Mặc định notebook được cấu hình để chạy gần đầy đủ tiêu chí. Nếu máy yếu, có thể giảm số dòng hoặc giảm số trial của Optuna. Nếu muốn chạy full dataset, đặt các biến `MAX_TRAIN_ROWS`, `MAX_VAL_ROWS`, `MAX_TEST_ROWS` bằng 0.


In [2]:
def find_project_root() -> Path:
    """Tìm thư mục gốc project dựa trên sự tồn tại của folder outputs/ hoặc data/."""
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "outputs").exists() or (parent / "data").exists():
            return parent
    return current

BASE_DIR = find_project_root()

PH03_OUTPUT_DIR = BASE_DIR / "outputs" / "ph03"
PH05_OUTPUT_DIR = BASE_DIR / "outputs" / "ph05"
MODEL_DIR = BASE_DIR / "models"

PH05_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PKL = PH03_OUTPUT_DIR / "processed_airglobal_features.pkl"
INPUT_CSV = PH03_OUTPUT_DIR / "processed_airglobal_features.csv"
FEATURE_CATALOG = PH03_OUTPUT_DIR / "feature_catalog.csv"

RANDOM_STATE = 42
TARGET_COL = "target_aqi_bucket"
SPLIT_COL = "split"

# Set 0 để dùng toàn bộ dữ liệu. Nếu máy yếu, có thể đặt 100_000 / 20_000 / 20_000.
MAX_TRAIN_ROWS = 0
MAX_VAL_ROWS = 0
MAX_TEST_ROWS = 0

# Stratified 5-Fold CV cho baseline. Set 0 để dùng toàn bộ train set, hoặc đặt 50_000 nếu máy yếu.
RUN_BASELINE_CV = True
BASELINE_CV_MAX_ROWS = 0

# Optuna tuning. Theo bảng phân công là 200 trials, nhưng có thể dùng 20 hoặc 50 nếu máy yếu.
OPTUNA_TRIALS = 20
OPTUNA_TIMEOUT = None

# Stacking Ensemble theo bảng phân công.
TRAIN_STACKING = True
STACKING_CV = 3

# Explainability và logging.
RUN_SHAP = True
RUN_LIME = True
RUN_PDP_ICE = True
RUN_MLFLOW = True
RUN_CALIBRATION = True

print("Project root:", BASE_DIR)
print("PH-03 input folder:", PH03_OUTPUT_DIR)
print("PH-05 output folder:", PH05_OUTPUT_DIR)
print("Model folder:", MODEL_DIR)


Project root: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL
PH-03 input folder: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL\outputs\ph03
PH-05 output folder: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL\outputs\ph05
Model folder: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL\models


## 3. Hàm tiện ích

Các hàm trong phần này hỗ trợ import thư viện optional, chuyển dữ liệu sparse sang dense khi cần, tạo one-hot encoder tương thích nhiều phiên bản scikit-learn và lấy sample theo tỷ lệ lớp.


In [3]:
def optional_import(module_name: str):
    try:
        return __import__(module_name)
    except Exception:
        return None


def to_dense_if_needed(X):
    if hasattr(X, "toarray"):
        return X.toarray()
    return X


def make_one_hot_encoder():
    """Tạo OneHotEncoder tương thích nhiều phiên bản scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def stratified_sample_indices(y: pd.Series, max_rows: int) -> List[int]:
    """Sample giữ tương đối tỷ lệ class để chạy nhanh khi chưa muốn dùng full dataset."""
    if max_rows <= 0 or len(y) <= max_rows:
        return list(y.index)

    parts = []
    for label, idx in y.groupby(y).groups.items():
        idx_list = list(idx)
        n_label = max(1, int(round(max_rows * len(idx_list) / len(y))))
        n_label = min(n_label, len(idx_list))
        sampled = pd.Series(idx_list).sample(n_label, random_state=RANDOM_STATE).tolist()
        parts.extend(sampled)

    if len(parts) > max_rows:
        parts = pd.Series(parts).sample(max_rows, random_state=RANDOM_STATE).tolist()

    return sorted(parts)


## 4. Đọc processed dataset từ PH-03

PH-05 chỉ sử dụng dataset đã được PH-03 tiền xử lý. Việc tách vai trò này giúp pipeline rõ ràng hơn: raw data đi qua PH-03 để tạo processed dataset, sau đó PH-05 mới dùng processed dataset để huấn luyện và đánh giá mô hình phân loại.


In [4]:
def load_processed_dataset() -> pd.DataFrame:
    if INPUT_PKL.exists():
        df = pd.read_pickle(INPUT_PKL)
        print(f"-> Loaded processed pickle: {INPUT_PKL.relative_to(BASE_DIR)}")
    elif INPUT_CSV.exists():
        df = pd.read_csv(INPUT_CSV)
        print(f"-> Loaded processed CSV: {INPUT_CSV.relative_to(BASE_DIR)}")
    else:
        raise FileNotFoundError("Không tìm thấy processed dataset. Hãy chạy PH-03 trước.")

    if TARGET_COL not in df.columns:
        raise ValueError(f"Không tìm thấy target column: {TARGET_COL}")
    if SPLIT_COL not in df.columns:
        raise ValueError(f"Không tìm thấy split column: {SPLIT_COL}")

    df = df.dropna(subset=[TARGET_COL, SPLIT_COL]).copy()
    df[SPLIT_COL] = df[SPLIT_COL].astype(str).str.lower().str.strip()

    print(f"-> Dataset: {len(df):,} rows | {df.shape[1]} columns")
    print("-> Split distribution:")
    print(df[SPLIT_COL].value_counts())
    print("-> Target distribution:")
    print(df[TARGET_COL].value_counts())
    return df


df = load_processed_dataset()


-> Loaded processed pickle: outputs\ph03\processed_airglobal_features.pkl
-> Dataset: 331,920 rows | 75 columns
-> Split distribution:
split
train    230500
val       50710
test      50710
Name: count, dtype: int64
-> Target distribution:
target_aqi_bucket
Moderate          124098
Good               89333
Unhealthy          54498
Satisfactory       53195
Very_Unhealthy      8045
Hazardous           2751
Name: count, dtype: int64


## 5. Chọn feature và kiểm soát data leakage

Vì nhãn AQI Bucket thường được suy ra trực tiếp từ AQI hoặc các biến tương đương, notebook loại bỏ chặt chẽ những cột có khả năng chứa đáp án như `aqi`, `bucket`, `category`, `target`, `label` và `class`. Đây là bước quan trọng để giảm nguy cơ kết quả bị thổi phồng do data leakage.


In [5]:
LEAKAGE_KEYWORDS = [
    "aqi",
    "bucket",
    "category",
    "label",
    "target",
    "class",
]

NON_FEATURE_COLUMNS = {
    "date",
    "datetime",
    "timestamp",
    "source_file",
    "file",
    SPLIT_COL,
    TARGET_COL,
}

# Nếu có feature nào chứa keyword nhưng chắc chắn không leakage, thêm vào danh sách này.
ALLOW_FEATURES_WITH_KEYWORDS: List[str] = []


def is_leakage_feature(col: str) -> bool:
    lower = col.lower().strip()
    if lower in [x.lower() for x in ALLOW_FEATURES_WITH_KEYWORDS]:
        return False
    if lower in {x.lower() for x in NON_FEATURE_COLUMNS}:
        return True
    return any(keyword in lower for keyword in LEAKAGE_KEYWORDS)


def get_feature_columns(df: pd.DataFrame):
    if FEATURE_CATALOG.exists():
        catalog = pd.read_csv(FEATURE_CATALOG)
        if "feature" in catalog.columns:
            candidate_features = [
                c for c in catalog["feature"].dropna().astype(str).tolist()
                if c in df.columns
            ]
        else:
            candidate_features = []
    else:
        candidate_features = []

    if not candidate_features:
        candidate_features = [c for c in df.columns if c not in NON_FEATURE_COLUMNS]

    safe_features = []
    removed_leakage = []
    removed_non_feature = []

    for col in candidate_features:
        if col in NON_FEATURE_COLUMNS:
            removed_non_feature.append(col)
            continue
        if is_leakage_feature(col):
            removed_leakage.append(col)
            continue
        safe_features.append(col)

    categorical_features = [
        c for c in safe_features
        if str(df[c].dtype) in ["object", "string", "category"]
    ]
    numeric_features = [c for c in safe_features if c not in categorical_features]

    leakage_report = pd.DataFrame({"removed_possible_leakage_feature": removed_leakage})
    leakage_report.to_csv(PH05_OUTPUT_DIR / "removed_possible_leakage_features.csv", index=False, encoding="utf-8-sig")

    print(f"-> Removed non-feature columns: {len(removed_non_feature)}")
    print(f"-> Removed possible leakage features: {len(removed_leakage)}")
    print(removed_leakage[:80])
    print(f"-> Numeric features: {len(numeric_features)}")
    print(f"-> Categorical features: {len(categorical_features)}")
    print(f"-> Total safe features: {len(safe_features)}")

    if not safe_features:
        raise ValueError("Không còn feature an toàn sau khi loại leakage. Cần kiểm tra lại PH-03 feature catalog.")

    return safe_features, numeric_features, categorical_features, removed_leakage


feature_cols, numeric_features, categorical_features, removed_leakage = get_feature_columns(df)


-> Removed non-feature columns: 0
-> Removed possible leakage features: 0
[]
-> Numeric features: 64
-> Categorical features: 4
-> Total safe features: 68


## 6. Chia train, validation và test

PH-03 đã tạo cột `split`, vì vậy PH-05 không chia lại dữ liệu từ đầu. Train set được dùng để huấn luyện, validation set được dùng để chọn mô hình và tuning, còn test set được giữ độc lập cho đánh giá cuối cùng.


In [6]:
def build_split_data(df: pd.DataFrame):
    encoder = LabelEncoder()
    encoder.fit(sorted(df[TARGET_COL].astype(str).unique()))

    data = {}
    for split_name, max_rows in [
        ("train", MAX_TRAIN_ROWS),
        ("val", MAX_VAL_ROWS),
        ("test", MAX_TEST_ROWS),
    ]:
        part = df[df[SPLIT_COL] == split_name].copy()
        if part.empty:
            raise ValueError(f"Split '{split_name}' không có dữ liệu. Hãy kiểm tra cột split từ PH-03.")

        y_raw = part[TARGET_COL].astype(str)
        selected_idx = stratified_sample_indices(y_raw, max_rows)
        part = part.loc[selected_idx].copy()

        X = part[feature_cols].copy()
        y = encoder.transform(part[TARGET_COL].astype(str))

        data[f"X_{split_name}"] = X
        data[f"y_{split_name}"] = y

    print(
        "-> Used rows:",
        f"train={len(data['X_train']):,},",
        f"val={len(data['X_val']):,},",
        f"test={len(data['X_test']):,}",
    )
    print("-> Classes:", list(encoder.classes_))
    return data, encoder


data, label_encoder = build_split_data(df)


-> Used rows: train=230,500, val=50,710, test=50,710
-> Classes: [np.str_('Good'), np.str_('Hazardous'), np.str_('Moderate'), np.str_('Satisfactory'), np.str_('Unhealthy'), np.str_('Very_Unhealthy')]


## 7. Tạo preprocessing transformer

Numeric features được chuẩn hóa bằng StandardScaler, còn categorical features được mã hóa bằng One-Hot Encoding. Pipeline được giữ nguyên trong artifact cuối để backend có thể load trực tiếp khi dự đoán.


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), numeric_features),
        ("cat", make_one_hot_encoder(), categorical_features),
    ],
    remainder="drop",
)


def make_pipeline(model):
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])

print("-> Preprocessing transformer created.")


-> Preprocessing transformer created.


## 8. Khai báo các mô hình baseline và boosting

Các mô hình được khai báo gồm Logistic Regression dạng SGD, Decision Tree, Random Forest, ExtraTrees, XGBoost và LightGBM. XGBoost và LightGBM chỉ được thêm vào danh sách nếu môi trường đã cài thư viện tương ứng.


In [8]:
def build_models() -> Dict[str, Pipeline]:
    models = {
        "LogisticRegression_SGD": make_pipeline(
            SGDClassifier(
                loss="log_loss",
                alpha=0.0005,
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )
        ),
        "DecisionTree": make_pipeline(
            DecisionTreeClassifier(
                max_depth=18,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )
        ),
        "RandomForest": make_pipeline(
            RandomForestClassifier(
                n_estimators=160,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
        "ExtraTrees": make_pipeline(
            ExtraTreesClassifier(
                n_estimators=160,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
    }

    xgb = optional_import("xgboost")
    if xgb is not None:
        models["XGBoost"] = make_pipeline(
            xgb.XGBClassifier(
                n_estimators=350,
                max_depth=8,
                learning_rate=0.06,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="multi:softprob",
                eval_metric="mlogloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        )
    else:
        print("Warning: xgboost is not installed. XGBoost skipped.")

    lgb = optional_import("lightgbm")
    if lgb is not None:
        models["LightGBM"] = make_pipeline(
            lgb.LGBMClassifier(
                n_estimators=450,
                learning_rate=0.05,
                num_leaves=63,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )
        )
    else:
        print("Warning: lightgbm is not installed. LightGBM skipped.")

    return models


models = build_models()
print("-> Models:", list(models.keys()))


-> Models: ['LogisticRegression_SGD', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM']


## 9. Baseline Stratified 5-Fold Cross Validation

Phần này đáp ứng yêu cầu đánh giá baseline bằng Stratified 5-Fold CV. Cross-validation được thực hiện trên train set để ước lượng độ ổn định của Logistic Regression, Decision Tree và Random Forest trước khi huấn luyện mô hình cuối.


In [9]:
def run_baseline_5fold_cv() -> pd.DataFrame:
    if not RUN_BASELINE_CV:
        print("-> Baseline 5-Fold CV disabled.")
        return pd.DataFrame()

    baseline_names = ["LogisticRegression_SGD", "DecisionTree", "RandomForest"]
    baseline_models = {name: models[name] for name in baseline_names if name in models}

    y_train_series = pd.Series(data["y_train"])
    selected_idx = stratified_sample_indices(y_train_series, BASELINE_CV_MAX_ROWS)
    X_cv = data["X_train"].iloc[selected_idx].copy()
    y_cv = data["y_train"][selected_idx]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for name, model in baseline_models.items():
        print(f"-> 5-Fold CV for {name}")
        scores = cross_validate(
            model,
            X_cv,
            y_cv,
            cv=cv,
            scoring={
                "accuracy": "accuracy",
                "macro_f1": "f1_macro",
                "weighted_f1": "f1_weighted",
            },
            n_jobs=-1,
            return_train_score=False,
        )
        rows.append({
            "model": name,
            "cv_rows": len(X_cv),
            "cv_accuracy_mean": scores["test_accuracy"].mean(),
            "cv_accuracy_std": scores["test_accuracy"].std(),
            "cv_macro_f1_mean": scores["test_macro_f1"].mean(),
            "cv_macro_f1_std": scores["test_macro_f1"].std(),
            "cv_weighted_f1_mean": scores["test_weighted_f1"].mean(),
            "cv_weighted_f1_std": scores["test_weighted_f1"].std(),
        })

    cv_df = pd.DataFrame(rows)
    cv_df.to_csv(PH05_OUTPUT_DIR / "baseline_5fold_cv_metrics.csv", index=False, encoding="utf-8-sig")
    return cv_df


baseline_cv_df = run_baseline_5fold_cv()
baseline_cv_df


-> 5-Fold CV for LogisticRegression_SGD
-> 5-Fold CV for DecisionTree
-> 5-Fold CV for RandomForest


,model,cv_rows,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,cv_weighted_f1_mean,cv_weighted_f1_std
0,LogisticRegression_SGD,230500,0.579753,0.051281,0.502037,0.066681,0.526883,0.081683
1,DecisionTree,230500,0.988074,0.000655,0.942234,0.002846,0.988270,0.000633
2,RandomForest,230500,0.931844,0.001866,0.889661,0.003831,0.932677,0.001806


## 10. Train và đánh giá các mô hình chính

Mỗi mô hình được huấn luyện trên train set, đánh giá trên validation set để lựa chọn mô hình và đánh giá trên test set để báo cáo kết quả cuối cùng. Notebook cũng lưu confusion matrix cho từng mô hình.


In [10]:
def evaluate_model(model_name: str, model, X, y, split_name: str) -> Dict[str, float]:
    pred = model.predict(X)
    precision, recall, _, _ = precision_recall_fscore_support(
        y, pred, average="macro", zero_division=0
    )
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y, pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "cohen_kappa": float(cohen_kappa_score(y, pred)),
    }


def save_confusion_matrix(model_name: str, model, X, y, split_name: str = "test"):
    pred = model.predict(X)
    labels = np.arange(len(label_encoder.classes_))
    cm = confusion_matrix(y, pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        title=f"Confusion Matrix - {model_name} ({split_name})",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=np.arange(len(label_encoder.classes_)),
        yticks=np.arange(len(label_encoder.classes_)),
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    fig.tight_layout()
    fig.savefig(PH05_OUTPUT_DIR / f"confusion_matrix_{model_name}_{split_name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


metrics_rows = []
trained_models: Dict[str, Any] = {}

print("-> Training baseline and boosting models...")
for name, model in models.items():
    print(f"-> Training {name}")
    model.fit(data["X_train"], data["y_train"])
    trained_models[name] = model

    metrics_rows.append(evaluate_model(name, model, data["X_val"], data["y_val"], "val"))
    metrics_rows.append(evaluate_model(name, model, data["X_test"], data["y_test"], "test"))
    save_confusion_matrix(name, model, data["X_test"], data["y_test"], "test")

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")
metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


-> Training baseline and boosting models...
-> Training LogisticRegression_SGD
-> Training DecisionTree
-> Training RandomForest
-> Training ExtraTrees
-> Training XGBoost
-> Training LightGBM


,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289
10,LightGBM,val,0.989174,0.954891,0.989107,0.963429,0.947053,0.985261
8,XGBoost,val,0.987379,0.952730,0.987249,0.965264,0.941409,0.982812
2,DecisionTree,val,0.988129,0.941551,0.988372,0.927842,0.958198,0.983859
4,RandomForest,val,0.933938,0.893597,0.934689,0.884957,0.906605,0.910892


## 11. Optuna tuning cho XGBoost

Phần này tối ưu siêu tham số của XGBoost bằng Optuna. Kết quả tốt nhất được lưu lại để phục vụ báo cáo và so sánh với mô hình mặc định.


In [11]:
def tune_xgboost_with_optuna() -> Optional[Pipeline]:
    if OPTUNA_TRIALS <= 0:
        print("-> Optuna disabled. Set OPTUNA_TRIALS > 0 to tune XGBoost.")
        return None

    optuna = optional_import("optuna")
    xgb = optional_import("xgboost")
    if optuna is None or xgb is None:
        print("Warning: optuna or xgboost is not installed. XGBoost tuning skipped.")
        return None

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "objective": "multi:softprob",
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        pipe = make_pipeline(xgb.XGBClassifier(**params))
        pipe.fit(data["X_train"], data["y_train"])
        pred = pipe.predict(data["X_val"])
        return f1_score(data["y_val"], pred, average="macro", zero_division=0)

    pruner = optuna.pruners.HyperbandPruner()
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=False)

    best_params = dict(study.best_params)
    best_params.update({
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    })

    best_model = make_pipeline(xgb.XGBClassifier(**best_params))
    best_model.fit(data["X_train"], data["y_train"])

    with open(PH05_OUTPUT_DIR / "optuna_xgboost_best_params.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2)

    print("-> Tuned XGBoost validation Macro F1:", study.best_value)
    return best_model


tuned_xgb = tune_xgboost_with_optuna()

if tuned_xgb is not None:
    trained_models["XGBoost_Optuna"] = tuned_xgb
    new_rows = [
        evaluate_model("XGBoost_Optuna", tuned_xgb, data["X_val"], data["y_val"], "val"),
        evaluate_model("XGBoost_Optuna", tuned_xgb, data["X_test"], data["y_test"], "test"),
    ]
    metrics_df = pd.concat([metrics_df, pd.DataFrame(new_rows)], ignore_index=True)
    metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")
    save_confusion_matrix("XGBoost_Optuna", tuned_xgb, data["X_test"], data["y_test"], "test")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


[I 2026-06-06 00:16:48,873] A new study created in memory with name: no-name-c3f04d33-6145-4921-8bdd-41c2f1987167
[I 2026-06-06 00:17:41,873] Trial 0 finished with value: 0.7825812666782546 and parameters: {'n_estimators': 555, 'max_depth': 4, 'learning_rate': 0.0010286207148652296, 'subsample': 0.8897025791694195, 'colsample_bytree': 0.7329173224231641, 'min_child_weight': 1.2981032945157347, 'gamma': 4.392186801359176, 'reg_alpha': 7.33113082892214e-06, 'reg_lambda': 0.01604152536179605}. Best is trial 0 with value: 0.7825812666782546.
[I 2026-06-06 00:19:27,357] Trial 1 finished with value: 0.9532517169328992 and parameters: {'n_estimators': 871, 'max_depth': 8, 'learning_rate': 0.009288526161038858, 'subsample': 0.8195166339367914, 'colsample_bytree': 0.6788657395946118, 'min_child_weight': 8.87319476273973, 'gamma': 4.392820686970455, 'reg_alpha': 2.547440877936065e-05, 'reg_lambda': 0.001315471365355328}. Best is trial 1 with value: 0.9532517169328992.
[I 2026-06-06 00:20:04,924]

-> Tuned XGBoost validation Macro F1: 0.9536971567318121


,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
13,XGBoost_Optuna,test,0.988523,0.958031,0.988439,0.970630,0.946683,0.984382
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289
10,LightGBM,val,0.989174,0.954891,0.989107,0.963429,0.947053,0.985261
12,XGBoost_Optuna,val,0.987537,0.953697,0.987394,0.969676,0.939708,0.983027
8,XGBoost,val,0.987379,0.952730,0.987249,0.965264,0.941409,0.982812


## 12. Optuna tuning cho LightGBM

Phần này tối ưu siêu tham số của LightGBM bằng Optuna. Đây là phần bổ sung để đáp ứng riêng tiêu chí LightGBM với Optuna tuning trong bảng phân công.


In [12]:
def tune_lightgbm_with_optuna() -> Optional[Pipeline]:
    if OPTUNA_TRIALS <= 0:
        print("-> Optuna disabled. Set OPTUNA_TRIALS > 0 to tune LightGBM.")
        return None

    optuna = optional_import("optuna")
    lgb = optional_import("lightgbm")
    if optuna is None or lgb is None:
        print("Warning: optuna or lightgbm is not installed. LightGBM tuning skipped.")
        return None

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbose": -1,
        }
        pipe = make_pipeline(lgb.LGBMClassifier(**params))
        pipe.fit(data["X_train"], data["y_train"])
        pred = pipe.predict(data["X_val"])
        return f1_score(data["y_val"], pred, average="macro", zero_division=0)

    pruner = optuna.pruners.HyperbandPruner()
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=False)

    best_params = dict(study.best_params)
    best_params.update({
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    })

    best_model = make_pipeline(lgb.LGBMClassifier(**best_params))
    best_model.fit(data["X_train"], data["y_train"])

    with open(PH05_OUTPUT_DIR / "optuna_lightgbm_best_params.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2)

    print("-> Tuned LightGBM validation Macro F1:", study.best_value)
    return best_model


tuned_lgbm = tune_lightgbm_with_optuna()

if tuned_lgbm is not None:
    trained_models["LightGBM_Optuna"] = tuned_lgbm
    new_rows = [
        evaluate_model("LightGBM_Optuna", tuned_lgbm, data["X_val"], data["y_val"], "val"),
        evaluate_model("LightGBM_Optuna", tuned_lgbm, data["X_test"], data["y_test"], "test"),
    ]
    metrics_df = pd.concat([metrics_df, pd.DataFrame(new_rows)], ignore_index=True)
    metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")
    save_confusion_matrix("LightGBM_Optuna", tuned_lgbm, data["X_test"], data["y_test"], "test")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


[I 2026-06-06 00:43:46,729] A new study created in memory with name: no-name-9f870c3a-c37e-48d8-829c-b94ec5829364
[I 2026-06-06 00:44:06,925] Trial 0 finished with value: 0.9510056332464275 and parameters: {'n_estimators': 239, 'learning_rate': 0.041206649536394815, 'num_leaves': 40, 'max_depth': 12, 'min_child_samples': 43, 'subsample': 0.6831767030656005, 'colsample_bytree': 0.952322242885058, 'reg_alpha': 4.1331821808190723e-07, 'reg_lambda': 0.030583389453363426}. Best is trial 0 with value: 0.9510056332464275.
[I 2026-06-06 00:44:37,333] Trial 1 finished with value: 0.9373488972918329 and parameters: {'n_estimators': 599, 'learning_rate': 0.014587217981906774, 'num_leaves': 152, 'max_depth': 5, 'min_child_samples': 87, 'subsample': 0.7291895501929326, 'colsample_bytree': 0.99308722030721, 'reg_alpha': 2.5710126742025146e-06, 'reg_lambda': 0.00426876794182273}. Best is trial 0 with value: 0.9510056332464275.
[I 2026-06-06 00:44:49,768] Trial 2 finished with value: 0.879473725922426

-> Tuned LightGBM validation Macro F1: 0.9560892432122964


,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
13,XGBoost_Optuna,test,0.988523,0.958031,0.988439,0.970630,0.946683,0.984382
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
15,LightGBM_Optuna,test,0.990515,0.956110,0.990470,0.962313,0.950252,0.987097
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289
14,LightGBM_Optuna,val,0.989430,0.956089,0.989367,0.963660,0.949017,0.985609
10,LightGBM,val,0.989174,0.954891,0.989107,0.963429,0.947053,0.985261


## 13. Stacking Ensemble

Stacking Ensemble kết hợp XGBoost, LightGBM, Random Forest và ExtraTrees làm base learners, sau đó sử dụng Logistic Regression làm meta learner. Mục tiêu là tận dụng ưu điểm của nhiều mô hình khác nhau thay vì phụ thuộc vào một mô hình đơn lẻ.


In [13]:
def train_stacking_if_enabled() -> Optional[Any]:
    if not TRAIN_STACKING:
        print("-> Stacking disabled. Set TRAIN_STACKING = True to train.")
        return None

    estimators = []

    if "XGBoost_Optuna" in trained_models:
        estimators.append(("xgb", trained_models["XGBoost_Optuna"]))
    elif "XGBoost" in trained_models:
        estimators.append(("xgb", trained_models["XGBoost"]))

    if "LightGBM_Optuna" in trained_models:
        estimators.append(("lgbm", trained_models["LightGBM_Optuna"]))
    elif "LightGBM" in trained_models:
        estimators.append(("lgbm", trained_models["LightGBM"]))

    if "RandomForest" in trained_models:
        estimators.append(("rf", trained_models["RandomForest"]))

    if "ExtraTrees" in trained_models:
        estimators.append(("extra", trained_models["ExtraTrees"]))

    if len(estimators) < 2:
        print("Warning: Not enough base models for stacking.")
        return None

    meta_learner = LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_learner,
        stack_method="predict_proba",
        cv=STACKING_CV,
        n_jobs=-1,
        passthrough=False,
    )

    print("-> Training Stacking Ensemble")
    stack.fit(data["X_train"], data["y_train"])
    return stack


stacking_model = train_stacking_if_enabled()

if stacking_model is not None:
    trained_models["StackingEnsemble"] = stacking_model
    new_rows = [
        evaluate_model("StackingEnsemble", stacking_model, data["X_val"], data["y_val"], "val"),
        evaluate_model("StackingEnsemble", stacking_model, data["X_test"], data["y_test"], "test"),
    ]
    metrics_df = pd.concat([metrics_df, pd.DataFrame(new_rows)], ignore_index=True)
    metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")
    save_confusion_matrix("StackingEnsemble", stacking_model, data["X_test"], data["y_test"], "test")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


-> Training Stacking Ensemble


,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
13,XGBoost_Optuna,test,0.988523,0.958031,0.988439,0.970630,0.946683,0.984382
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
15,LightGBM_Optuna,test,0.990515,0.956110,0.990470,0.962313,0.950252,0.987097
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
17,StackingEnsemble,test,0.987419,0.937291,0.987765,0.918535,0.963513,0.982908
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289
14,LightGBM_Optuna,val,0.989430,0.956089,0.989367,0.963660,0.949017,0.985609


## 14. Chọn best model theo validation Macro F1

Validation set được sử dụng để chọn mô hình tốt nhất, sau đó test set được dùng để báo cáo kết quả cuối. Cách làm này giúp tránh việc lựa chọn mô hình dựa trực tiếp trên test set.


In [14]:
val_metrics = metrics_df[metrics_df["split"] == "val"].copy()
best_model_name = val_metrics.sort_values("macro_f1", ascending=False).iloc[0]["model"]
best_model = trained_models[best_model_name]

print("-> Best model by validation Macro F1:", best_model_name)

test_metrics = metrics_df[metrics_df["split"] == "test"].sort_values("macro_f1", ascending=False)
test_metrics.to_csv(PH05_OUTPUT_DIR / "test_metrics_sorted.csv", index=False, encoding="utf-8-sig")
test_metrics


-> Best model by validation Macro F1: LightGBM_Optuna


,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
13,XGBoost_Optuna,test,0.988523,0.958031,0.988439,0.970630,0.946683,0.984382
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
15,LightGBM_Optuna,test,0.990515,0.956110,0.990470,0.962313,0.950252,0.987097
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
17,StackingEnsemble,test,0.987419,0.937291,0.987765,0.918535,0.963513,0.982908
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289


## 15. Kiểm tra xác suất dự đoán và calibration

Các mô hình boosting có thể cho xác suất quá tự tin dù nhãn dự đoán đúng. Calibration được dùng để làm xác suất mềm và đáng tin hơn, phù hợp hơn khi hiển thị trên app/backend. Cell này lưu thêm mô hình calibrated để nhóm có thể dùng trong production nếu cần.


In [18]:
# True: chạy calibration để làm xác suất mềm hơn
# False: bỏ qua calibration, tiếp tục các phần SHAP/LIME/PDP/MLflow

ENABLE_CALIBRATION = False

def save_probability_report(model_name: str, model, suffix: str):
    if not hasattr(model, "predict_proba"):
        print(f"Warning: {model_name} has no predict_proba. Probability report skipped.")
        return None

    proba = model.predict_proba(data["X_test"])
    pred = np.argmax(proba, axis=1)
    max_proba = np.max(proba, axis=1)

    report = pd.DataFrame({
        "true_label": label_encoder.inverse_transform(data["y_test"]),
        "pred_label": label_encoder.inverse_transform(pred),
        "max_probability": max_proba,
        "is_correct": pred == data["y_test"],
    })

    summary = pd.DataFrame({
        "model": [model_name],
        "suffix": [suffix],
        "mean_max_probability": [float(report["max_probability"].mean())],
        "median_max_probability": [float(report["max_probability"].median())],
        "p90_max_probability": [float(report["max_probability"].quantile(0.90))],
        "p99_max_probability": [float(report["max_probability"].quantile(0.99))],
        "accuracy": [float((report["is_correct"]).mean())],
        "log_loss": [float(log_loss(data["y_test"], proba))],
    })

    report.to_csv(
        PH05_OUTPUT_DIR / f"probability_report_{suffix}.csv",
        index=False,
        encoding="utf-8-sig"
    )
    summary.to_csv(
        PH05_OUTPUT_DIR / f"probability_summary_{suffix}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return summary


def calibrate_best_model_if_enabled():
    if not ENABLE_CALIBRATION:
        print("-> Calibration disabled.")
        return None

    if not hasattr(best_model, "predict_proba"):
        print("Warning: best model does not support predict_proba. Calibration skipped.")
        return None

    print("-> Probability report before calibration")
    raw_summary = save_probability_report(best_model_name, best_model, "raw")
    display(raw_summary)

    calibrated_model = None

    try:
        from sklearn.frozen import FrozenEstimator

        calibrated_model = CalibratedClassifierCV(
            estimator=FrozenEstimator(best_model),
            method="sigmoid"
        )

        print("-> Using FrozenEstimator calibration mode")

    except Exception as exc_frozen:
        print(f"Warning: FrozenEstimator mode failed: {type(exc_frozen).__name__}: {exc_frozen}")

        try:
            calibrated_model = CalibratedClassifierCV(
                estimator=best_model,
                method="sigmoid",
                cv="prefit"
            )

            print("-> Using estimator + cv='prefit' calibration mode")

        except Exception as exc_prefit:
            print(f"Warning: Prefit calibration failed: {type(exc_prefit).__name__}: {exc_prefit}")
            print("-> Calibration skipped. Raw probability report was still saved.")
            return None

    try:
        print("-> Calibrating probability with validation set")
        calibrated_model.fit(data["X_val"], data["y_val"])

        print("-> Probability report after calibration")
        calibrated_summary = save_probability_report(
            best_model_name + "_Calibrated",
            calibrated_model,
            "calibrated"
        )
        display(calibrated_summary)

        return calibrated_model

    except Exception as exc_fit:
        print(f"Warning: Calibration fit failed: {type(exc_fit).__name__}: {exc_fit}")
        print("-> Calibration skipped. You can continue the notebook without calibration.")
        return None


calibrated_best_model = calibrate_best_model_if_enabled()

if calibrated_best_model is not None:
    trained_models[best_model_name + "_Calibrated"] = calibrated_best_model

-> Calibration disabled.


## 16. Feature importance

Feature importance giúp xác định những biến có ảnh hưởng mạnh nhất đến mô hình. Nếu mô hình tốt nhất là tree-based model, notebook sử dụng `feature_importances_`; nếu không, notebook dùng permutation importance làm phương án thay thế.


In [19]:
def get_transformed_feature_names(model) -> List[str]:
    if isinstance(model, Pipeline):
        pre = model.named_steps.get("preprocess")
        if pre is not None:
            try:
                return list(pre.get_feature_names_out())
            except Exception:
                names = []
                names.extend([f"num__{c}" for c in numeric_features])
                names.extend([f"cat__{c}" for c in categorical_features])
                return names
    return feature_cols


def save_feature_importance(model_name: str, model) -> pd.DataFrame:
    model_step = model.named_steps.get("model") if isinstance(model, Pipeline) else model
    feature_names = get_transformed_feature_names(model)

    if hasattr(model_step, "feature_importances_"):
        importances = np.asarray(model_step.feature_importances_)
        n = min(len(importances), len(feature_names))
        result = pd.DataFrame({
            "feature": feature_names[:n],
            "importance": importances[:n],
        }).sort_values("importance", ascending=False)
    else:
        print("-> Using permutation importance fallback.")
        perm = permutation_importance(
            model,
            data["X_test"].sample(min(3000, len(data["X_test"])), random_state=RANDOM_STATE),
            data["y_test"][:min(3000, len(data["y_test"]))],
            n_repeats=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            scoring="f1_macro",
        )
        result = pd.DataFrame({
            "feature": feature_cols,
            "importance": perm.importances_mean,
        }).sort_values("importance", ascending=False)

    result.to_csv(PH05_OUTPUT_DIR / "best_model_feature_importance.csv", index=False, encoding="utf-8-sig")

    top20 = result.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top20["feature"], top20["importance"])
    ax.set_title(f"Top 20 Feature Importance - {model_name}")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    fig.savefig(PH05_OUTPUT_DIR / "best_model_feature_importance_top20.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return result


feature_importance_df = save_feature_importance(best_model_name, best_model)
feature_importance_df.head(20)


,feature,importance
0,num__pm25,14041
62,num__wind_speed_lag_3,12321
61,num__wind_speed_lag_1,11711
15,num__industry_growth,11102
53,num__o3_lag_3,11029
63,num__wind_speed_lag_12,10998
52,num__o3_lag_1,10415
31,num__benzene_toluene_ratio,10269
8,num__o3,10208
59,num__humidity_lag_3,10205


## 17. SHAP Explainability

SHAP được dùng để giải thích mức đóng góp của từng feature vào kết quả dự đoán. Nếu SHAP không tương thích với mô hình tốt nhất hoặc môi trường chưa cài thư viện, notebook vẫn giữ feature importance làm kết quả giải thích thay thế.


In [20]:
def try_save_shap_summary(model_name: str, model):
    if not RUN_SHAP:
        print("-> SHAP disabled.")
        return

    shap = optional_import("shap")
    if shap is None:
        print("Warning: shap is not installed. Install by: pip install shap")
        return

    try:
        if not isinstance(model, Pipeline):
            print("Warning: SHAP in this cell expects a sklearn Pipeline. Skipped.")
            return

        pre = model.named_steps["preprocess"]
        model_step = model.named_steps["model"]

        if not hasattr(model_step, "predict_proba"):
            print("Warning: model step has no predict_proba. SHAP skipped.")
            return

        X_sample = data["X_test"].sample(min(1000, len(data["X_test"])), random_state=RANDOM_STATE)
        X_transformed = pre.transform(X_sample)
        X_transformed = to_dense_if_needed(X_transformed)
        feature_names = get_transformed_feature_names(model)

        explainer = shap.TreeExplainer(model_step)
        shap_values = explainer.shap_values(X_transformed)

        plt.figure()
        shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(PH05_OUTPUT_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
        plt.close()
        print("-> SHAP summary saved.")
    except Exception as exc:
        print(f"Warning: SHAP skipped due to {type(exc).__name__}: {exc}")


try_save_shap_summary(best_model_name, best_model)


-> SHAP summary saved.


<Figure size 640x480 with 0 Axes>

## 18. LIME Explainability

LIME được dùng để giải thích một dự đoán cụ thể. Kết quả được lưu dưới dạng file HTML để có thể mở trực tiếp trên trình duyệt và chụp hình đưa vào báo cáo.


In [33]:
def try_save_lime_explanation(model_name: str, model):
    lime = optional_import("lime")
    if lime is None:
        print("Warning: lime is not installed. Install by: pip install lime")
        return

    try:
        from lime.lime_tabular import LimeTabularExplainer

        pre = model.named_steps["preprocess"]
        model_step = model.named_steps["model"]

        X_train_sample = data["X_train"].sample(
            min(5000, len(data["X_train"])),
            random_state=RANDOM_STATE
        )

        X_test_sample = data["X_test"].copy()

        X_train_transformed = pre.transform(X_train_sample)
        X_test_transformed = pre.transform(X_test_sample)

        feature_names = get_transformed_feature_names(model)
        class_names = list(label_encoder.classes_)

        proba = model_step.predict_proba(X_test_transformed)
        pred_idx = np.argmax(proba, axis=1)
        pred_labels = label_encoder.inverse_transform(pred_idx)

        preferred_classes = [
            "Hazardous",
            "Very_Unhealthy",
            "Very Unhealthy",
            "Unhealthy",
            "Poor",
            "Moderate"
        ]

        selected_row = None
        selected_class_name = None
        selected_class_index = None

        for cls in preferred_classes:
            if cls in class_names:
                cls_index = class_names.index(cls)
                candidate_indices = np.where(pred_idx == cls_index)[0]

                if len(candidate_indices) > 0:
                    selected_row = int(candidate_indices[0])
                    selected_class_name = cls
                    selected_class_index = cls_index
                    break

        if selected_row is None:
            confidence = np.max(proba, axis=1)
            selected_row = int(np.argmin(confidence))
            selected_class_index = int(pred_idx[selected_row])
            selected_class_name = class_names[selected_class_index]

        print("-> LIME selected row:", selected_row)
        print("-> LIME predicted class:", selected_class_name)
        print("-> LIME predicted probability:", proba[selected_row, selected_class_index])

        explainer = LimeTabularExplainer(
            training_data=np.asarray(X_train_transformed),
            feature_names=feature_names,
            class_names=class_names,
            mode="classification",
            discretize_continuous=True,
            random_state=RANDOM_STATE
        )

        explanation = explainer.explain_instance(
            data_row=np.asarray(X_test_transformed[selected_row]),
            predict_fn=model_step.predict_proba,
            labels=[selected_class_index],
            num_features=12
        )

        html_path = PH05_OUTPUT_DIR / "lime_explanation_sample.html"
        explanation.save_to_file(str(html_path))

        lime_rows = explanation.as_list(label=selected_class_index)
        lime_df = pd.DataFrame(lime_rows, columns=["feature_rule", "lime_weight"])
        lime_df.to_csv(
            PH05_OUTPUT_DIR / "lime_explanation_sample.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print("-> LIME explanation saved:", html_path.relative_to(BASE_DIR))
        display(lime_df)

    except Exception as exc:
        print(f"Warning: LIME skipped due to {type(exc).__name__}: {exc}")


try_save_lime_explanation(best_model_name, best_model)

-> LIME selected row: 24
-> LIME predicted class: Hazardous
-> LIME predicted probability: 0.9999988578752733


## 19. PDP và ICE Plot

PDP mô tả ảnh hưởng trung bình của một feature đến dự đoán, còn ICE thể hiện ảnh hưởng đó trên từng mẫu riêng lẻ. Notebook ưu tiên chọn các feature numeric quan trọng nhất để vẽ PDP/ICE.


In [29]:
from sklearn.inspection import PartialDependenceDisplay

def choose_pdp_target_class():
    class_names = list(label_encoder.classes_)

    preferred_targets = [
        "Unhealthy",
        "Very Unhealthy",
        "Unhealthy for Sensitive Groups",
        "Moderate",
        "Hazardous"
    ]

    for name in preferred_targets:
        if name in class_names:
            return name, class_names.index(name)

    return class_names[-1], len(class_names) - 1


def choose_pdp_features():
    preferred_features = [
        "pm25",
        "pm10",
        "no2",
        "so2",
        "co",
        "o3",
        "humidity",
        "wind_speed"
    ]

    selected = []

    for feature in preferred_features:
        if feature in feature_cols and feature not in categorical_features:
            selected.append(feature)

    if len(selected) == 0:
        for feature in feature_cols:
            if feature not in categorical_features:
                selected.append(feature)
            if len(selected) >= 5:
                break

    return selected[:5]


def save_pdp_ice_fixed(model_name: str, model):
    target_class_name, target_class_index = choose_pdp_target_class()
    selected_features = choose_pdp_features()

    print("-> PDP/ICE target class:", target_class_name)
    print("-> PDP/ICE target index:", target_class_index)
    print("-> PDP/ICE features:", selected_features)

    X_pdp = data["X_test"].sample(
        min(1500, len(data["X_test"])),
        random_state=RANDOM_STATE
    )

    for feature in selected_features:
        print("-> Saving PDP/ICE for:", feature)

        fig, ax = plt.subplots(figsize=(7, 5))

        PartialDependenceDisplay.from_estimator(
            model,
            X_pdp,
            features=[feature],
            target=target_class_index,
            kind="average",
            grid_resolution=30,
            ax=ax
        )

        ax.set_title(f"PDP - {feature} - target: {target_class_name}")
        ax.set_ylabel(f"Predicted probability of {target_class_name}")

        safe_feature_name = (
            feature.replace("/", "_")
                   .replace("\\", "_")
                   .replace(" ", "_")
                   .replace(":", "_")
        )

        fig.tight_layout()
        fig.savefig(
            PH05_OUTPUT_DIR / f"pdp_{safe_feature_name}_{target_class_name}.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.close(fig)

    print("-> Fixed PDP plots saved.")


save_pdp_ice_fixed(best_model_name, best_model)

-> PDP/ICE target class: Unhealthy
-> PDP/ICE target index: 4
-> PDP/ICE features: ['pm25', 'pm10', 'no2', 'so2', 'co']
-> Saving PDP/ICE for: pm25
-> Saving PDP/ICE for: pm10
-> Saving PDP/ICE for: no2
-> Saving PDP/ICE for: so2
-> Saving PDP/ICE for: co
-> Fixed PDP plots saved.


## 20. MLflow logging

MLflow được sử dụng để ghi lại tham số thực nghiệm, chỉ số đánh giá và các artifact quan trọng như metrics, confusion matrix, feature importance, SHAP và LIME. Nếu môi trường chưa cài MLflow, cell này sẽ bỏ qua nhưng notebook vẫn chạy tiếp.


In [30]:
def try_log_mlflow():
    if not RUN_MLFLOW:
        print("-> MLflow logging disabled.")
        return

    mlflow = optional_import("mlflow")
    if mlflow is None:
        print("Warning: mlflow is not installed. Install by: pip install mlflow")
        return

    try:
        mlflow.set_experiment("AirGlobal_PH05_Classification")

        with mlflow.start_run(run_name=f"best_model_{best_model_name}"):
            mlflow.log_param("best_model", best_model_name)
            mlflow.log_param("target_col", TARGET_COL)
            mlflow.log_param("n_features", len(feature_cols))
            mlflow.log_param("n_numeric_features", len(numeric_features))
            mlflow.log_param("n_categorical_features", len(categorical_features))
            mlflow.log_param("train_rows", len(data["X_train"]))
            mlflow.log_param("val_rows", len(data["X_val"]))
            mlflow.log_param("test_rows", len(data["X_test"]))
            mlflow.log_param("optuna_trials", OPTUNA_TRIALS)
            mlflow.log_param("train_stacking", TRAIN_STACKING)
            mlflow.log_param("removed_leakage_features", len(removed_leakage))

            best_test_rows = metrics_df[(metrics_df["model"] == best_model_name) & (metrics_df["split"] == "test")]
            if len(best_test_rows) > 0:
                best_test = best_test_rows.iloc[0].to_dict()
                for key, value in best_test.items():
                    if isinstance(value, (int, float, np.integer, np.floating)):
                        mlflow.log_metric(key, float(value))

            artifact_files = [
                "model_metrics.csv",
                "test_metrics_sorted.csv",
                "baseline_5fold_cv_metrics.csv",
                "best_model_feature_importance.csv",
                "best_model_feature_importance_top20.png",
                "removed_possible_leakage_features.csv",
                "shap_summary.png",
                "lime_explanation_sample.html",
                "probability_summary_raw.csv",
                "probability_summary_calibrated.csv",
                f"confusion_matrix_{best_model_name}_test.png",
            ]

            for file_name in artifact_files:
                file_path = PH05_OUTPUT_DIR / file_name
                if file_path.exists():
                    mlflow.log_artifact(str(file_path))

            for file_path in PH05_OUTPUT_DIR.glob("pdp_ice_*.png"):
                mlflow.log_artifact(str(file_path))

            print("-> MLflow logging completed.")
    except Exception as exc:
        print(f"Warning: MLflow logging skipped due to {type(exc).__name__}: {exc}")


try_log_mlflow()


-> MLflow logging completed.


## 21. Lưu production artifacts

Các artifact quan trọng gồm pipeline tốt nhất, label encoder, danh sách feature đầu vào, metadata mô hình và mô hình calibrated nếu có. Backend có thể load các file này để dự đoán AQI Bucket.


In [34]:
with open(MODEL_DIR / "best_aqi_classifier_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)

if calibrated_best_model is not None:
    with open(MODEL_DIR / "best_aqi_classifier_calibrated.pkl", "wb") as f:
        pickle.dump(calibrated_best_model, f)

with open(MODEL_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

with open(MODEL_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

metadata = {
    "best_model": best_model_name,
    "target_col": TARGET_COL,
    "split_col": SPLIT_COL,
    "n_features": len(feature_cols),
    "n_numeric_features": len(numeric_features),
    "n_categorical_features": len(categorical_features),
    "removed_possible_leakage_features": len(removed_leakage),
    "train_rows": int(len(data["X_train"])),
    "val_rows": int(len(data["X_val"])),
    "test_rows": int(len(data["X_test"])),
    "baseline_cv": bool(RUN_BASELINE_CV),
    "optuna_trials": int(OPTUNA_TRIALS),
    "train_stacking": bool(TRAIN_STACKING),
    "run_shap": bool(RUN_SHAP),
    "run_lime": bool(RUN_LIME),
    "run_pdp_ice": bool(RUN_PDP_ICE),
    "run_mlflow": bool(RUN_MLFLOW),
    "run_calibration": bool(RUN_CALIBRATION),
    "has_calibrated_model": calibrated_best_model is not None,
    "classes": list(label_encoder.classes_),
}

with open(PH05_OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("-> Saved production artifacts.")
print("Model folder:", MODEL_DIR.relative_to(BASE_DIR))
print("Output folder:", PH05_OUTPUT_DIR.relative_to(BASE_DIR))


-> Saved production artifacts.
Model folder: models
Output folder: outputs\ph05


## 22. Tóm tắt kết quả và file output

Cell cuối cùng in danh sách output để kiểm tra nhanh trước khi push lên GitHub. Không nên push các file model hoặc dataset lớn hơn 100MB lên GitHub.


In [35]:
print("Output files:")
for path in sorted(PH05_OUTPUT_DIR.glob("*")):
    print("-", path.relative_to(BASE_DIR))

print("\nModel files:")
for path in sorted(MODEL_DIR.glob("*")):
    print("-", path.relative_to(BASE_DIR), f"({path.stat().st_size / (1024 * 1024):.2f} MB)")

print("\nFinal test metrics:")
display(metrics_df[metrics_df["split"] == "test"].sort_values("macro_f1", ascending=False))

print("\nBaseline 5-Fold CV metrics:")
display(baseline_cv_df)


Output files:
- outputs\ph05\baseline_5fold_cv_metrics.csv
- outputs\ph05\best_model_feature_importance.csv
- outputs\ph05\best_model_feature_importance_top20.png
- outputs\ph05\confusion_matrix_DecisionTree_test.png
- outputs\ph05\confusion_matrix_ExtraTrees_test.png
- outputs\ph05\confusion_matrix_LightGBM_Optuna_test.png
- outputs\ph05\confusion_matrix_LightGBM_test.png
- outputs\ph05\confusion_matrix_LogisticRegression_SGD_test.png
- outputs\ph05\confusion_matrix_RandomForest_test.png
- outputs\ph05\confusion_matrix_StackingEnsemble_test.png
- outputs\ph05\confusion_matrix_XGBoost_Optuna_test.png
- outputs\ph05\confusion_matrix_XGBoost_test.png
- outputs\ph05\lime_explanation_sample.html
- outputs\ph05\model_metadata.json
- outputs\ph05\model_metrics.csv
- outputs\ph05\optuna_lightgbm_best_params.json
- outputs\ph05\optuna_xgboost_best_params.json
- outputs\ph05\pdp_co_Unhealthy.png
- outputs\ph05\pdp_no2_Unhealthy.png
- outputs\ph05\pdp_pm10_Unhealthy.png
- outputs\ph05\pdp_pm25_U

,model,split,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa
13,XGBoost_Optuna,test,0.988523,0.958031,0.988439,0.970630,0.946683,0.984382
11,LightGBM,test,0.990515,0.957259,0.990476,0.963458,0.951445,0.987098
9,XGBoost,test,0.988306,0.956992,0.988225,0.968140,0.946816,0.984087
15,LightGBM_Optuna,test,0.990515,0.956110,0.990470,0.962313,0.950252,0.987097
3,DecisionTree,test,0.988977,0.939300,0.989283,0.921849,0.962321,0.985024
17,StackingEnsemble,test,0.987419,0.937291,0.987765,0.918535,0.963513,0.982908
5,RandomForest,test,0.937764,0.895293,0.938420,0.885225,0.909018,0.916010
7,ExtraTrees,test,0.839519,0.783831,0.842277,0.747051,0.847588,0.785791
1,LogisticRegression_SGD,test,0.788030,0.741059,0.785409,0.790526,0.715452,0.711289



Baseline 5-Fold CV metrics:


,model,cv_rows,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,cv_weighted_f1_mean,cv_weighted_f1_std
0,LogisticRegression_SGD,230500,0.579753,0.051281,0.502037,0.066681,0.526883,0.081683
1,DecisionTree,230500,0.988074,0.000655,0.942234,0.002846,0.988270,0.000633
2,RandomForest,230500,0.931844,0.001866,0.889661,0.003831,0.932677,0.001806
